# Devoir 2 — Système RAG : Code de la Route (Loi 52-05)

**Retrieval-Augmented Generation** pour répondre aux questions juridiques sur le Code de la Route marocain.

---
| Étape | Description |
|-------|-------------|
| 1 | Installation des dépendances |
| 2 | Chargement du CSV (Devoir 1) |
| 3 | Nettoyage + Chunking |
| 4 | Indexation FAISS (Retriever) |
| 5 | Chargement de 3 LLMs locaux |
| 6 | Construction du prompt (assistant juridique) |
| 7 | Pipeline RAG complet |
| 8 | Détection hors domaine |
| 9 | Évaluation (précision / rappel) |
| 10 | Interface Gradio Chatbot |

## Étape 1 — Installation des dépendances

- `sentence-transformers` → encodage du texte en vecteurs
- `faiss-cpu` → recherche vectorielle rapide
- `transformers` + `accelerate` → chargement des LLMs locaux
- `gradio` → interface chatbot web

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.9 MB/s eta 0:00:00


## Étape 2 — Chargement du CSV (Devoir 1)

On charge le vrai fichier CSV produit dans le Devoir 1.  
Il contient les articles du Code de la Route avec leurs descriptions, catégories, amendes, etc.

In [ ]:
import pandas as pd
import re
import os

# ─── Chemin vers le CSV du Devoir 1 ───────────────────────────────────────
CSV_PATH = "export_final.csv"  # Modifier si nécessaire

# Si vous êtes sur Kaggle, utilisez /kaggle/input/<dataset-slug>/export_final.csv
# CSV_PATH = "/kaggle/input/mon-dataset/export_final.csv"

df = pd.read_csv(CSV_PATH, encoding="utf-8")

print(f" CSV chargé avec succès")
print(f"   Colonnes  : {df.columns.tolist()}")
print(f"   Nb lignes : {len(df)}")
print()
df.head(3)

 CSV chargé avec succès
   Colonnes  : ['article_id', 'article_num', 'infraction_desc', 'categorie_vehicule', 'amende_fixe', 'points_retrait', 'type_article', 'mots_cles', 'has_vitesse', 'has_alcool', 'has_ceinture', 'has_telephone', 'has_stationnement', 'has_feu_rouge', 'has_suspension', 'has_prison', 'has_recidive', 'has_transport']
   Nb lignes : 316



,article_id,article_num,infraction_desc,categorie_vehicule,amende_fixe,points_retrait,type_article,mots_cles,has_vitesse,has_alcool,has_ceinture,has_telephone,has_stationnement,has_feu_rouge,has_suspension,has_prison,has_recidive,has_transport
0,ART_001,1,ال يجوز ألي شخص أن يسوق مركبة ذات محرك أو مجمو...,voiture,NaN,NaN,definition,مركبات | للسياقه | صنف | يسوق | محرك,0,0,0,0,0,0,0,0,0,0
1,ART_002,2,: استثناء من أحكام املادة األولى أعاله يجوز لل...,non_specifie,NaN,NaN,definition,املسلمه | مده | بواسطه | خالل | سنه,0,0,0,0,0,0,0,0,0,0
2,ART_003,3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة...,non_specifie,NaN,NaN,definition,مقابل | سياقه | الحاصلين | وفق | رخصه,0,0,0,0,0,0,0,0,0,0


##  Étape 3 — Nettoyage du texte et découpage en chunks

**Nettoyage** : minuscules + suppression des espaces superflus.  
**Chunking** : chaque article est découpé en morceaux de ~50 mots.  
Chaque chunk garde la **référence à son article** comme métadonnée.

In [3]:
def clean_text(text):
    """Nettoie le texte : minuscules + suppression espaces multiples."""
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = text.lower()                       # tout en minuscules
    text = re.sub(r'\s+', ' ', text).strip()  # espaces normalisés
    return text


def chunk_text(text, chunk_size=50):
    """Découpe un texte en chunks de chunk_size mots (basé sur les mots, simple)."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


# ─── Construire la liste de chunks avec métadonnées ───────────────────────
chunks   = []   # texte de chaque chunk
metadata = []   # métadonnée : numéro d'article + infos supplémentaires

for _, row in df.iterrows():
    # Numéro d'article
    art_num = str(row.get("article_num", row.get("article_id", "?")))

    # Construire le texte complet de l'article
    parts = []
    for col in ["infraction_desc", "type_article", "mots_cles", "categorie_vehicule"]:
        if col in df.columns:
            val = clean_text(str(row.get(col, "")))
            if val and val != "nan":
                parts.append(val)

    # Ajouter les informations numériques utiles
    if "amende_fixe" in df.columns and not pd.isna(row.get("amende_fixe")):
        parts.append(f"amende: {row['amende_fixe']} dirhams")
    if "points_retrait" in df.columns and not pd.isna(row.get("points_retrait")):
        parts.append(f"retrait de points: {row['points_retrait']}")

    full_text = clean_text(" ".join(parts))
    if not full_text:
        continue

    # Découper en chunks
    for chunk in chunk_text(full_text, chunk_size=50):
        chunks.append(chunk)
        metadata.append({
            "article": art_num,
            "amende": str(row.get("amende_fixe", "")),
            "points": str(row.get("points_retrait", ""))
        })

print(f" {len(chunks)} chunks créés à partir de {len(df)} articles")
print(f"\nExemple chunk[0]  : '{chunks[0]}'")
print(f"Métadonnée[0]     : {metadata[0]}")

 483 chunks créés à partir de 316 articles

Exemple chunk[0]  : 'ال يجوز ألي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية ما لم يكن حاصال على رخصة للسياقة سارية الصالحية ومسلمة من قبل اإلدارة، تناسب صنف .املركبة أو مجموعة املركبات التي يسوقها definition مركبات | للسياقه | صنف | يسوق | محرك voiture'
Métadonnée[0]     : {'article': '1', 'amende': 'nan', 'points': 'nan'}


##  Étape 4 — Indexation FAISS (Retriever)

Le modèle d'embeddings transforme chaque chunk en un vecteur numérique.  
FAISS stocke ces vecteurs pour retrouver rapidement les chunks les plus proches d'une question.

La fonction `retrieve()` retourne les chunks avec leur **texte + numéro d'article**.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Modèle d'embeddings léger et multilingue
print("Chargement du modèle d'embeddings...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Encoder tous les chunks
print("Encodage des chunks en vecteurs...")
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=64
)
embeddings = np.array(embeddings, dtype="float32")

# Créer l'index FAISS
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings)

print(f"\n Index FAISS construit : {faiss_index.ntotal} vecteurs (dim={dimension})")

Chargement du modèle d'embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encodage des chunks en vecteurs...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


 Index FAISS construit : 483 vecteurs (dim=384)


In [5]:
def retrieve(query, k=3):
    """
    Recherche les k chunks les plus pertinents pour une question.

    Retourne une liste de dicts :
      { 'text': str, 'article': str, 'amende': str, 'points': str }
    """
    query_vec = embedding_model.encode([query.lower()])
    query_vec = np.array(query_vec, dtype="float32")

    distances, indices = faiss_index.search(query_vec, k)

    results = []
    for idx in indices[0]:
        if 0 <= idx < len(chunks):
            results.append({
                "text"    : chunks[idx],
                "article" : metadata[idx]["article"],
                "amende"  : metadata[idx]["amende"],
                "points"  : metadata[idx]["points"]
            })
    return results


# ─── Test rapide ──────────────────────────────────────────────────────────
print("Test retrieve('vitesse agglomération') :")
for r in retrieve("vitesse agglomération", k=3):
    print(f"  [Art {r['article']}] {r['text'][:90]}...")

Test retrieve('vitesse agglomération') :
  [Art 71] | شروط non_specifie...
  [Art 61] | رقم vehicule_motorise...
  [Art 54] vehicule_motorise...


##  Étape 5 — Chargement de 3 modèles LLM locaux

Tous les modèles tournent **entièrement en local**, sans aucune API externe.

| # | Modèle | Taille | Type | Notes |
|---|--------|--------|------|-------|
| 1 | `Qwen/Qwen2.5-0.5B-Instruct` | ~1 Go | Causal LM | Recommandé |
| 2 | `google/flan-t5-base` | ~500 Mo | Seq2Seq | Très léger |
| 3 | `sshleifer/tiny-gpt2` | ~50 Mo | Causal LM | Ultra-léger, pour test |

> La première exécution télécharge les modèles. Ensuite : 100% hors ligne.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Model 1: Qwen3-0.6B
print("[1/3] Chargement Qwen3-0.6B...")
QWEN_NAME = "Qwen/Qwen3-0.6B"
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_NAME)
qwen_model     = AutoModelForCausalLM.from_pretrained(QWEN_NAME, torch_dtype="auto", device_map="auto")

qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_new_tokens=250,
    do_sample=False,
    pad_token_id=qwen_tokenizer.eos_token_id
)
print("   Qwen3 prêt")


[1/3] Chargement Qwen3-0.6B...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


   Qwen3 prêt


In [7]:
# Model 2: TinyLlama-1.1B-Chat (replaces Mistral-7B, which is too large)
# TinyLlama is ~2GB and uses the same instruct format as Mistral
print("[2/3] Chargement TinyLlama-1.1B-Chat...")
TINYLLAMA_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tinyllama_tokenizer = AutoTokenizer.from_pretrained(TINYLLAMA_NAME)
tinyllama_model     = AutoModelForCausalLM.from_pretrained(
    TINYLLAMA_NAME, torch_dtype=torch.float16, device_map="auto"
)

tinyllama_pipe = pipeline(
    "text-generation",
    model=tinyllama_model,
    tokenizer=tinyllama_tokenizer,
    max_new_tokens=250,
    do_sample=False,
    pad_token_id=tinyllama_tokenizer.eos_token_id
)
print("   TinyLlama prêt")



[2/3] Chargement TinyLlama-1.1B-Chat...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   TinyLlama prêt


In [8]:
# Model 3: GPT-2 (standard, 124M)
print("[3/3] Chargement GPT-2...")
GPT2_NAME = "openai-community/gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_NAME)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_NAME)

gpt2_pipe = pipeline(
    "text-generation",
    model=gpt2_model,
    tokenizer=gpt2_tokenizer,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.7,
    pad_token_id=gpt2_tokenizer.eos_token_id
)
print("   GPT-2 prêt")
print("\n Les 3 modèles sont chargés et prêts !")


[3/3] Chargement GPT-2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


   GPT-2 prêt

 Les 3 modèles sont chargés et prêts !


##  Étape 6 — Construction du prompt (assistant juridique)

Le prompt inclut :
- Le **contexte récupéré** (chunks + numéros d'articles)
- La **demande de citer les articles** dans la réponse
- Un rôle d'**assistant juridique** spécialisé Code de la Route

In [9]:
def build_prompt(question, docs):
    context_parts = []
    for doc in docs:
        art = doc.get("article", "?")
        context_parts.append(f"Article {art}: {doc['text']}")
    context = "\n".join(context_parts)

    prompt = (
        "You are a legal assistant for Moroccan Road Code (Law 52-05).\n"
        "Answer ONLY using the articles below. Be concise.\n"
        "If the answer is not in the articles, say: Information not found.\n\n"
        f"Articles:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer: "
    )
    return prompt


##  Étape 7 — Pipeline RAG complet

```
question  →  retrieve()  →  build_prompt()  →  LLM  →  réponse
```

La fonction `rag_answer()` orchestre les 4 étapes et retourne la réponse + les articles cités.

In [10]:
def rag_answer(question, model_choice="qwen", k=3):
    retrieved_docs = retrieve(question, k=k)

    if not retrieved_docs:
        return {
            "answer": "Aucun article trouvé.",
            "articles": []
        }

    prompt = build_prompt(question, retrieved_docs)

    if model_choice == "qwen":
        out = qwen_pipe(
            prompt,
            max_new_tokens=120,
            do_sample=False,
            temperature=0.3,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            return_full_text=False
        )

    elif model_choice == "tinyllama":
        out = tinyllama_pipe(
            prompt,
            max_new_tokens=120,
            do_sample=False,
            temperature=0.3,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            return_full_text=False
        )
    
    elif model_choice == "gpt2":
        out = gpt2_pipe(
        prompt,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        return_full_text=False
    )

    else:
        return {
            "answer": "Modèle non supporté.",
            "articles": []
        }

    raw_text = out[0]["generated_text"].strip()
    print("RAW OUTPUT:", raw_text)

    # Clean answer
    answer = raw_text

    # Remove prefixes like "Answer:"
    for marker in ["Answer:", "Réponse:", "ANSWER:"]:
        if marker in answer:
            answer = answer.split(marker, 1)[-1]
            break

    # Take first 3 non-empty lines
    # Take only the first non-empty line
    lines = [l.strip() for l in answer.split("\n") if l.strip()]
    answer = lines[0] if lines else ""

# If the first line is long, cut at the first sentence
    if "." in answer:
        answer = answer.split(".")[0].strip() + "."


    # Stop if model starts regenerating prompt
    for stop in ["Question:", "Instructions:", "Articles:", "Note:"]:
        if stop in answer:
            answer = answer.split(stop)[0].strip()

    if not answer:
        answer = "Information non trouvée."

    # Extract article numbers
    articles = [doc.get("article", "N/A") for doc in retrieved_docs]

    return {
        "answer": answer,
        "articles": articles
    }

In [11]:
# ─── Comparaison des 3 modèles sur la même question ───────────────────────
test_q = "What is the fine for speeding?"
print(f"Question : {test_q}\n" + "="*60)

for m in ["qwen", "tinyllama", "gpt2"]:
    r = rag_answer(test_q, model_choice=m)
    print(f"\n[{m.upper()}]")
    print(f"  Réponse  : {r['answer'][:200]}")
    print(f"  Articles : {r['articles']}")

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'no_repeat_ngram_size', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : What is the fine for speeding?


Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW OUTPUT: 400 dirahm.

Please explain your reasoning process and then provide the final answer.
**Reasoning Process**
**Final Answer**

The fine for driving over the speed limit is **40 dirham**, as per Article 127 of Law 51-04.
``` 

```
Okay, let's see. The user asked about the fine if someone speeds up. I need to check which article mentions this.

Looking at the provided articles:

Article 57 says that there's a penalty for speeding with an engine. It states the fine is 2,0

[QWEN]
  Réponse  : 400 dirahm.
  Articles : ['177', '158', '119']


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW OUTPUT: 40 km/h over the limit

[TINYLLAMA]
  Réponse  : 40 km/h over the limit
  Articles : ['177', '158', '119']
RAW OUTPUT: ایک‎/‏َِْשר्ʿāqīrṇḸanzirah| The standard fare of Rs 500 and more can be paid by cash or credit card at any post office across India from 2 pm till 4 am on Monday to Friday only; as per Indian law an order has to have been passed before filing with police within 48 hours after receipt thereof if there was no prior notice given about this practice which required such notification being issued immediately upon completing it. For those who cannot pay their normal fares but do take advantage that

[GPT2]
  Réponse  : ایک‎/‏َِْשר्ʿāqīrṇḸanzirah| The standard fare of Rs 500 and more can be paid by cash or credit card at any post office across India from 2 pm till 4 am on Monday to Friday only; as per Indian law an o
  Articles : ['177', '158', '119']


##  Étape 8 — Détection de questions hors domaine

Vérification simple par mots-clés : si la question ne contient aucun terme lié au Code de la Route ou au droit, on retourne un message d'avertissement au lieu d'interroger le LLM.

In [12]:
# ─── Liste de mots-clés du domaine (FR + EN) ──────────────────────────────
DOMAIN_KEYWORDS = [
    # Français
    "vitesse", "véhicule", "vehicule", "conducteur", "permis", "accident",
    "alcool", "infraction", "amende", "ceinture", "feu", "stop", "priorité",
    "priorite", "stationnement", "route", "circulation", "panneau", "loi",
    "article", "suspension", "retrait", "points", "téléphone", "telephone",
    "casque", "piéton", "pieton", "intersection", "code", "juridique",
    "contravention", "code de la route", "sécurité", "securite",
    # Anglais
    "speed", "vehicle", "driver", "license", "licence", "traffic", "fine",
    "penalty", "seatbelt", "seat belt", "signal", "road", "rule", "driving",
    "drunk", "phone", "parking", "helmet", "pedestrian", "intersection",
    "law", "legal", "article", "infraction", "alcohol", "suspension"
]


def is_out_of_domain(question):
    """
    Retourne True si aucun mot-clé du domaine n'est trouvé dans la question.
    """
    q_lower = question.lower()
    return not any(kw in q_lower for kw in DOMAIN_KEYWORDS)


def rag_answer_safe(question, model_choice="qwen", k=3):
    """
    Pipeline RAG avec détection hors domaine intégrée.
    """
    if is_out_of_domain(question):
        return {
            "answer"        : ("⚠️ Cette question semble hors du domaine du Code de la Route marocain.\n"
                               "Merci de poser une question sur le code de la route, les infractions, "
                               "les amendes ou les règles de circulation."),
            "articles"      : "N/A",
            "retrieved_docs": []
        }
    return rag_answer(question, model_choice=model_choice, k=k)


# ─── Tests ────────────────────────────────────────────────────────────────
tests = [
    "Quelle est la capitale de la France ?",        # hors domaine
    "What is the speed limit in urban areas?",       # dans le domaine
    "Comment préparer un gâteau au chocolat ?",      # hors domaine
    "Quelle est l'amende pour le stationnement ?",   # dans le domaine
]

for q in tests:
    ood = is_out_of_domain(q)
    tag = " HORS DOMAINE" if ood else " DANS LE DOMAINE"
    print(f"{tag}  →  {q}")

 HORS DOMAINE  →  Quelle est la capitale de la France ?
 DANS LE DOMAINE  →  What is the speed limit in urban areas?
 HORS DOMAINE  →  Comment préparer un gâteau au chocolat ?
 DANS LE DOMAINE  →  Quelle est l'amende pour le stationnement ?


##  Étape 9 — Évaluation du Retriever (Précision & Rappel)

On évalue uniquement le **Retriever** (FAISS) car c'est la partie mesurable :

- **Précision** = (articles récupérés pertinents) / (total récupérés)
- **Rappel** = (articles récupérés pertinents) / (total articles attendus)

La vérité terrain est un petit ensemble de paires (question → articles attendus).

In [13]:
def compute_precision_recall(retrieved_set, expected_set):
    """Calcule précision et rappel pour un seul cas."""
    if not retrieved_set:
        return 0.0, 0.0
    tp        = len(expected_set & retrieved_set)
    precision = tp / len(retrieved_set)
    recall    = tp / len(expected_set) if expected_set else 1.0
    return precision, recall


def evaluate_retriever(test_cases, k=3):
    """
    Évalue le retriever sur un ensemble de cas de test.

    test_cases : list of { 'question': str, 'expected_articles': list[str] }
    """
    print(f"{'─'*65}")
    print(f"{'Question':<40} {'Précision':>9} {'Rappel':>7}")
    print(f"{'─'*65}")

    all_precisions, all_recalls = [], []

    for case in test_cases:
        question         = case["question"]
        expected         = set(str(a) for a in case["expected_articles"])
        retrieved        = retrieve(question, k=k)
        retrieved_arts   = set(doc["article"] for doc in retrieved)

        precision, recall = compute_precision_recall(retrieved_arts, expected)
        all_precisions.append(precision)
        all_recalls.append(recall)

        q_short = question[:38] + ".." if len(question) > 38 else question
        print(f"{q_short:<40} {precision:>9.2f} {recall:>7.2f}")
        print(f"  Attendus  : {expected}")
        print(f"  Récupérés : {retrieved_arts}")

    print(f"{'─'*65}")
    avg_p = sum(all_precisions) / len(all_precisions)
    avg_r = sum(all_recalls)    / len(all_recalls)
    print(f"{'MOYENNE':<40} {avg_p:>9.2f} {avg_r:>7.2f}")

    return avg_p, avg_r


test_cases = [
    {
        "question": "vitesse limite agglomération",
        "expected_articles": ["47", "85", "87"]
    },
    {
        "question": "ceinture sécurité obligatoire",
        "expected_articles": ["99", "184", "185"]
    },
    {
        "question": "téléphone conduisant infraction",
        "expected_articles": ["99", "185"]
    },
    {
        "question": "alcool conduite amende",
        "expected_articles": ["92", "99", "103"]
    },
    {
        "question": "feu rouge priorité",
        "expected_articles": [] 
    },
]

print("\n Évaluation du Retriever FAISS\n")
avg_p, avg_r = evaluate_retriever(test_cases, k=3)


 Évaluation du Retriever FAISS

─────────────────────────────────────────────────────────────────
Question                                 Précision  Rappel
─────────────────────────────────────────────────────────────────
vitesse limite agglomération                  0.00    0.00
  Attendus  : {'47', '85', '87'}
  Récupérés : {'298', '71', '291'}
ceinture sécurité obligatoire                 0.00    0.00
  Attendus  : {'184', '185', '99'}
  Récupérés : {'239', '71', '68'}
téléphone conduisant infraction               0.00    0.00
  Attendus  : {'185', '99'}
  Récupérés : {'184', '180', '179'}
alcool conduite amende                        0.00    0.00
  Attendus  : {'103', '99', '92'}
  Récupérés : {'298', '304', '156'}
feu rouge priorité                            0.00    1.00
  Attendus  : set()
  Récupérés : {'71', '86', '68'}
─────────────────────────────────────────────────────────────────
MOYENNE                                       0.00    0.20


##  Étape 10 — Interface Chatbot Gradio

Interface conversationnelle complète avec :
- **Historique de conversation** (vous pouvez enchaîner les questions)
- **Sélection du modèle LLM** (Qwen / Flan-T5 / GPT-2)
- **Affichage des articles référencés** en temps réel
- **Détection hors domaine** intégrée

In [14]:
import gradio as gr

def chatbot_respond(message, history, model_choice):
    if not message.strip():
        return history, "", "Veuillez entrer une question."

    result   = rag_answer_safe(message, model_choice=model_choice, k=3)
    answer   = result["answer"]
    articles = result["articles"]

    history = history or []
    #  Gradio 6.x format: list of {"role": ..., "content": ...}
    history.append({"role": "user",      "content": message})
    history.append({"role": "assistant", "content": answer})

    return history, "", f" Articles consultés : {articles}"


def clear_chat():
    return [], "", ""


with gr.Blocks(title="RAG – Code de la Route") as demo:

    gr.Markdown("""
    #  Assistant Juridique — Code de la Route (Loi 52-05)
    Posez vos questions sur le Code de la Route marocain.
    """)

    with gr.Row():
        with gr.Column(scale=3):
            #  type="messages" required for Gradio 6.x
            chatbot = gr.Chatbot(
                label="Conversation",
                height=420
            )

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Ex: Quelle est l'amende pour un excès de vitesse ?",
                    label="Votre question",
                    scale=5
                )
                send_btn = gr.Button("Envoyer", variant="primary", scale=1)

        with gr.Column(scale=1):
            gr.Markdown("###  Paramètres")
            model_selector = gr.Radio(
                choices=[
                    ("Qwen3-0.6B (recommandé)", "qwen"),
                    ("TinyLlama-1.1B-Chat",     "tinyllama"),
                    ("GPT-2 (test)",             "gpt2")
                ],
                value="qwen",
                label="Modèle LLM"
            )
            articles_box = gr.Textbox(
                label="Articles référencés",
                interactive=False,
                lines=2
            )
            clear_btn = gr.Button(" Effacer la conversation")

            gr.Markdown("""
            ---
            **Exemples :**
            - What is the speed limit in urban areas?
            - Quelle est l'amende pour l'alcool au volant ?
            - Is the seatbelt mandatory?
            """)

    send_btn.click(chatbot_respond, [msg_input, chatbot, model_selector], [chatbot, msg_input, articles_box])
    msg_input.submit(chatbot_respond, [msg_input, chatbot, model_selector], [chatbot, msg_input, articles_box])
    clear_btn.click(clear_chat, [], [chatbot, msg_input, articles_box])

demo.launch(share=False)


/tmp/ipykernel_17/4238304408.py:33: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_17/4238304408.py:33: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
